# Plant-ID: distil BioCLIP into a phone-sized student (Colab)

Trains the MobileCLIP2-S0 student and exports `plantid_v1.onnx` without touching the
garden app's database. Everything comes from a bundle built on the laptop by
`ml/data/build_plantid_colab_bundle.py`. Background: `ml/docs/PLANT_ID_MODEL.md`.

**Before running:**
1. *Runtime → Change runtime type → T4 GPU.*
2. Upload `plantid_colab_bundle.zip` (~2.5 GB) to **My Drive/plantid/**.

Checkpoints go to `My Drive/plantid/` after every epoch. If the session drops,
reconnect and re-run all cells: training resumes from the last epoch.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU: Runtime -> Change runtime type -> T4 GPU'

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/plantid'
# Unzip to local disk: reading 55k small files straight from Drive is very slow.
!mkdir -p /content/bundle && cp "$DRIVE/plantid_colab_bundle.zip" /content/ && unzip -q -o /content/plantid_colab_bundle.zip -d /content/bundle
!cat /content/bundle/meta.json

In [ ]:
!rm -rf /content/gardening_app && git clone -q --depth 1 -b plantid-vision https://github.com/gccarno/gardening_app.git /content/gardening_app
!pip install -q open_clip_torch timm onnx onnxruntime onnxscript
%cd /content/gardening_app

In [ ]:
# Teacher prototypes: BioCLIP text tower on the GPU (laptop ships image prototypes only).
!python ml/data/build_plantid_colab_bundle.py --finish /content/bundle


## Smoke test (~2 min)
256 photos, 1 epoch, written to a scratch folder. It catches setup problems before the long run.

In [ ]:
!python ml/training/train_plantid_student.py --bundle /content/bundle --out-dir /content/smoke     --student mobileclip2_s0 --epochs 1 --max-images 256 --workers 2 --tag smoke

## Full run (~1–2 h on a T4)
`--resume` is a no-op the first time, then continues from `My Drive/plantid/..._last.pt`
after a disconnect. Watch `calib top-1` per epoch: the pilot reached 4.1% after 2 short
epochs, and the teacher (BioCLIP) scores 43.7%.

In [ ]:
!python ml/training/train_plantid_student.py --bundle /content/bundle --out-dir "$DRIVE"     --student mobileclip2_s0 --epochs 15 --workers 2 --tag v1 --resume

## Export: int8 ONNX + calibration + prototypes
Fails without writing anything if int8 and fp32 agree on fewer than 97% of eval top-1 predictions.

In [ ]:
!python ml/training/export_plantid_onnx.py --student-ckpt "$DRIVE/student_mobileclip2_s0_v1.pt"     --bundle /content/bundle --out-dir "$DRIVE/export" --version v1
!ls -la "$DRIVE/export"

Download **`My Drive/plantid/export/`**: `plantid_v1.onnx`, `plantid_v1.json` and
`plantid_v1_prototypes.npz`. Hand them back to Claude Code to commit and load into Neon.